In [2]:
from datasets import load_dataset

# Load the SST-2 dataset from the GLUE benchmark
dataset = load_dataset("nyu-mll/glue", "sst2")

# Inspect the structure and a sample
print(dataset)
print("\nFirst training example:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

C:\Users\ASUS\anaconda3\envs\fewshot-sentiment\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\datasets--nyu-mll--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\ASUS\anaconda3\envs\fewshot-sentiment\lib\site-packages\huggingface_hub\file_download.py:149: UserW

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

First training example:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Extract text and labels
X_train = dataset["train"]["sentence"]
y_train = dataset["train"]["label"]
X_val = dataset["validation"]["sentence"]
y_val = dataset["validation"]["label"]

# 1. Vectorize the text using TF-IDF
# Limiting max_features is a common practice to avoid overfitting
tfidf_vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

# 2. Train a Logistic Regression classifier
# Setting a random_state ensures reproducibility
classifier = LogisticRegression(random_state=42, max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

# 3. Evaluate on the validation set
y_pred = classifier.predict(X_val_tfidf)
accuracy = accuracy_score(y_val, y_pred)

print(f"Baseline Validation Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["Negative", "Positive"]))

Baseline Validation Accuracy: 0.8188

Classification Report:
              precision    recall  f1-score   support

    Negative       0.84      0.78      0.81       428
    Positive       0.80      0.86      0.83       444

    accuracy                           0.82       872
   macro avg       0.82      0.82      0.82       872
weighted avg       0.82      0.82      0.82       872

